In [1]:
#load dependencies 
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/cifar10_experiment_task1_2")

# Task 1.1

In [2]:
# Device (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:08<00:00, 20.2MB/s] 


## Task 1.2

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

        self.activation = nn.LeakyReLU()

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))

        x = x.view(-1, 64 * 8 * 8)

        x = self.activation(self.fc1(x))
        x = self.fc2(x)

        return x

# Initialize model
model = SimpleCNN().to(device)

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [5]:
epochs = 50

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(trainloader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    writer.add_scalar("Loss/train", avg_loss, epoch)

Epoch 1, Loss: 2.0373
Epoch 2, Loss: 1.6970
Epoch 3, Loss: 1.5043
Epoch 4, Loss: 1.3769
Epoch 5, Loss: 1.2873
Epoch 6, Loss: 1.2185
Epoch 7, Loss: 1.1551
Epoch 8, Loss: 1.1007
Epoch 9, Loss: 1.0494
Epoch 10, Loss: 1.0018
Epoch 11, Loss: 0.9565
Epoch 12, Loss: 0.9151
Epoch 13, Loss: 0.8734
Epoch 14, Loss: 0.8340
Epoch 15, Loss: 0.7985
Epoch 16, Loss: 0.7594
Epoch 17, Loss: 0.7267
Epoch 18, Loss: 0.6898
Epoch 19, Loss: 0.6557
Epoch 20, Loss: 0.6223
Epoch 21, Loss: 0.5877
Epoch 22, Loss: 0.5552
Epoch 23, Loss: 0.5192
Epoch 24, Loss: 0.4871
Epoch 25, Loss: 0.4532
Epoch 26, Loss: 0.4230
Epoch 27, Loss: 0.3859
Epoch 28, Loss: 0.3558
Epoch 29, Loss: 0.3244
Epoch 30, Loss: 0.2892
Epoch 31, Loss: 0.2607
Epoch 32, Loss: 0.2317
Epoch 33, Loss: 0.2056
Epoch 34, Loss: 0.1822
Epoch 35, Loss: 0.1525
Epoch 36, Loss: 0.1333
Epoch 37, Loss: 0.1148
Epoch 38, Loss: 0.0946
Epoch 39, Loss: 0.0803
Epoch 40, Loss: 0.0626
Epoch 41, Loss: 0.0474
Epoch 42, Loss: 0.0485
Epoch 43, Loss: 0.0297
Epoch 44, Loss: 0.02

In [6]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

writer.add_scalar("Accuracy/test", accuracy)

writer.close()

Test Accuracy: 70.89%
